In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
torch.manual_seed(42)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/sample data/fashion-mnist_train.csv")
df.head()

In [ ]:
df.shape

In [ ]:
# Create a 4x4 grid of images
fig, axes = plt.subplots(2, 3, figsize=(8, 8))
fig. suptitle("First 6 Images", fontsize=16)

# Plot the first 16 images from the dataset
for i, ax in enumerate(axes.flat):
    img = df.iloc[i, 1:].values.reshape(28, 28) # Reshape to 28x28
    ax.imshow(img) # Display in grayscale
    ax.axis('off') # Remove axis for a cleaner look
    ax.set_title(f"Label: {df.iloc[i, 0]}") # Show the label

plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust layout to fit the title
plt.show()

In [ ]:
X = df.drop("label", axis = 1).to_numpy()
y = df["label"].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42)

X_train = X_train/255
X_test = X_test/255

In [ ]:
class customdata(Dataset):
    def __init__(self, features, labels):

        self.features = torch.tensor(features, dtype= torch.float32)
        self.labels = torch.tensor(labels, dtype= torch.long)

    def __len__(self):
        return self.features.shape[0]
    def __getitem__(self, index):
        return self.features[index], self.labels[index]


In [ ]:
train_dataset = customdata(X_train, y_train)

test_dataset = customdata(X_test, y_test)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory= True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True, pin_memory= True)

In [ ]:
len(train_dataloader)

#### Model

In [ ]:
class MyNN(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layers, nuron_per_layer):

        super().__init__()

        layer = []

        for num_hidden_layer in range(num_hidden_layers):
            layer.append(nn.Linear(input_dim, nuron_per_layer))
            layer.append(nn.BatchNorm1d(nuron_per_layer))
            layer.append(nn.ReLU())
            layer.append(nn.Dropout(0.3))
            input_dim = nuron_per_layer
        layer.append(nn.Linear(input_dim, output_dim))

        self.model = nn.Sequential(*layer)    # * ----> for unpacking List

    def forward(self, x):
        return self.model(x)

##### Objective Functions

In [ ]:
def objective(trial):

    # next layer Hyperparameters
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1,5)
    nuron_per_layer = trial.suggest_int("nuron_per_layer", 8, 128, step = 8)

    # model initial
    input_dim = X_train.shape[1]
    output_dim = len(np.unique(y_train))

    model = MyNN(input_dim, output_dim, num_hidden_layers, nuron_per_layer)
    model.to(device)

    # Paramms
    lr = 0.01
    epochs = 5

    # Optimizer selection
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr = lr, weight_decay= 1e-4)

    # training Loop
    for epoch in range (epochs):


        for batch_features, batch_labels in train_dataloader:

            #Moving Features and labes to GPU
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    
            # forward pass
            output = model(batch_features)

            # losss calculation
            loss = criterion(output, batch_labels)

            # gradient to 0
            optimizer.zero_grad()

            # Back Pass
            loss.backward()


            #updating grads
            optimizer.step()

    # Evalution
    model.eval()

    total = 0
    correct = 0

    with torch.no_grad():
        for batch_features, batch_labels in test_dataloader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)


            output = model(batch_features)

            val, predicted_index = torch.max(output, 1)    # dim 1 wise searching|

            total += batch_labels.shape[0]

            correct = correct+(predicted_index == batch_labels).sum().item()

            accurecy = correct/total
        return (accurecy)


In [ ]:
! pip install optuna


In [ ]:
import optuna

In [ ]:
study = optuna.create_study(direction= "maximize")

In [ ]:
study.optimize(objective, n_trials=10)

In [ ]:
study.best_value

In [ ]:
study.best_params